# 🎯 ACCOUNT 2: Text-Heavy Ensemble

**Strategy:** Deep text/NLP analysis with 3 text models  
**Expected:** 50-54% SMAPE  
**Runtime:** 3 hours on Colab T4 GPU  

---

## 🚀 Setup:
1. **Enable GPU:** Runtime → Change runtime type → T4 GPU → Save
2. **Upload data:** sample_train.csv + sample_test.csv
3. **Run all:** Runtime → Run all (Ctrl+F9)
4. **Wait 3 hours** - Auto-download submissions!

## 📊 This Ensemble:
- 📝 **Text:** BERT (768) + RoBERTa (768) + CLIP (512) = **2048 features**
- 🖼️ **Vision:** ResNet50 (2048) = **2048 features**
- ⚙️ **Advanced:** 25 engineered features
- 🤖 **ML Models:** XGBoost + LightGBM + CatBoost
- **Total:** 4,121 features focusing on TEXT QUALITY!

In [ ]:
# GPU Check
import torch
print("🔥 ACCOUNT 2: Text-Heavy Ensemble")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB\n")
else:
    raise Exception("❌ Enable GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
# Install packages (3-5 min)
print("📦 Installing packages...\n")
!pip install -q transformers torch torchvision timm pillow
!pip install -q xgboost lightgbm catboost scikit-learn
!pip install -q pandas numpy requests tqdm scipy
print("✅ Packages installed!\n")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from transformers import (
    BertTokenizer, BertModel,
    RobertaTokenizer, RobertaModel,
    CLIPProcessor, CLIPModel
)
from PIL import Image
import requests
from io import BytesIO
from sklearn.model_selection import KFold
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from tqdm import tqdm
import warnings
import gc
from google.colab import files
import time

warnings.filterwarnings('ignore')
device = torch.device('cuda')
print("✅ Libraries loaded!\n")

In [ ]:
# Upload dataset
print("📤 Upload sample_train.csv and sample_test.csv:\n")
uploaded = files.upload()
print("\n✅ Files uploaded!\n")

In [ ]:
# Load data
print("📂 Loading data...\n")
train_df = pd.read_csv('sample_train.csv')
test_df = pd.read_csv('sample_test.csv')
print(f"✅ Train: {train_df.shape}")
print(f"✅ Test: {test_df.shape}\n")

In [ ]:
# Text Extractor - 3 MODELS!
print("📝 Loading 3 text models (this is our strength!)...\n")

class TextExtractor:
    def __init__(self, device):
        self.device = device
        
        print("  [1/3] BERT (768-dim)...")
        self.bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.bert_model = BertModel.from_pretrained('bert-base-uncased')
        self.bert_model.eval().to(device)
        
        print("  [2/3] RoBERTa (768-dim)...")
        self.roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
        self.roberta_model = RobertaModel.from_pretrained('roberta-base')
        self.roberta_model.eval().to(device)
        
        print("  [3/3] CLIP Text (512-dim)...")
        self.clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        self.clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
        self.clip_model.eval().to(device)
        
        print("\n✅ Text models loaded → 2048 features!\n")
    
    def extract(self, text):
        with torch.no_grad():
            # BERT
            bert_inputs = self.bert_tokenizer(text, return_tensors='pt', truncation=True,
                                             padding='max_length', max_length=128)
            bert_inputs = {k: v.to(self.device) for k, v in bert_inputs.items()}
            bert_feat = self.bert_model(**bert_inputs).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            
            # RoBERTa
            roberta_inputs = self.roberta_tokenizer(text, return_tensors='pt', truncation=True,
                                                   padding='max_length', max_length=128)
            roberta_inputs = {k: v.to(self.device) for k, v in roberta_inputs.items()}
            roberta_feat = self.roberta_model(**roberta_inputs).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            
            # CLIP
            clip_inputs = self.clip_processor(text=[text], return_tensors='pt', truncation=True,
                                             padding=True, max_length=77)
            clip_inputs = {k: v.to(self.device) for k, v in clip_inputs.items()}
            clip_feat = self.clip_model.get_text_features(**clip_inputs).squeeze().cpu().numpy()
        
        return np.concatenate([bert_feat, roberta_feat, clip_feat])

text_extractor = TextExtractor(device)

In [ ]:
# Extract text features (1.5-2 hours)
print("📝 Extracting TEXT features (our competitive advantage!)...")
print(f"   Processing {len(train_df) + len(test_df)} texts...")
print(f"   ⏱️ ETA: 1.5-2 hours\n")

start = time.time()

# Train
train_text_features = []
for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Train Text"):
    features = text_extractor.extract(str(row['catalog_content']))
    train_text_features.append(features)
    if idx % 1000 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

train_text_df = pd.DataFrame(train_text_features, columns=[f'text_{i}' for i in range(2048)])
print(f"✅ Train text: {train_text_df.shape}")

# Test
test_text_features = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Text"):
    features = text_extractor.extract(str(row['catalog_content']))
    test_text_features.append(features)
    if idx % 1000 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

test_text_df = pd.DataFrame(test_text_features, columns=[f'text_{i}' for i in range(2048)])
print(f"✅ Test text: {test_text_df.shape}")
print(f"⏱️ Text time: {(time.time()-start)/3600:.1f} hours\n")

del text_extractor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Vision Extractor - 1 model (lightweight)
print("🖼️ Loading ResNet50 (single vision model)...\n")

class VisionExtractor:
    def __init__(self, device):
        self.device = device
        resnet = models.resnet50(pretrained=True)
        self.model = nn.Sequential(*list(resnet.children())[:-1])
        self.model.eval().to(device)
        
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        self.success = 0
        self.failed = 0
        print("✅ ResNet50 loaded → 2048 features!\n")
    
    def download_image(self, url):
        try:
            r = requests.get(url, timeout=5)
            img = Image.open(BytesIO(r.content)).convert('RGB')
            self.success += 1
            return img
        except:
            self.failed += 1
            return Image.new('RGB', (224, 224), color='gray')
    
    def extract(self, image_url):
        img = self.download_image(image_url)
        img_tensor = self.transform(img).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            features = self.model(img_tensor).squeeze().cpu().numpy()
        
        return features

vision_extractor = VisionExtractor(device)

In [ ]:
# Extract vision features (40-50 min)
print("🖼️ Extracting VISION features (quick!)...")
print(f"   ⏱️ ETA: 40-50 minutes\n")

# Train
train_vision_features = []
for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Train Vision"):
    features = vision_extractor.extract(row['image_link'])
    train_vision_features.append(features)
    if idx % 1000 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

train_vision_df = pd.DataFrame(train_vision_features, columns=[f'vision_{i}' for i in range(2048)])
print(f"✅ Train vision: {train_vision_df.shape}")

# Test
vision_extractor.success = 0
vision_extractor.failed = 0
test_vision_features = []
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Vision"):
    features = vision_extractor.extract(row['image_link'])
    test_vision_features.append(features)
    if idx % 1000 == 0 and idx > 0:
        gc.collect()
        torch.cuda.empty_cache()

test_vision_df = pd.DataFrame(test_vision_features, columns=[f'vision_{i}' for i in range(2048)])
print(f"✅ Test vision: {test_vision_df.shape}\n")

del vision_extractor
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Advanced features
def create_features(df):
    features = pd.DataFrame()
    features['text_len'] = df['catalog_content'].str.len()
    features['word_count'] = df['catalog_content'].str.split().str.len()
    features['avg_word_len'] = features['text_len'] / (features['word_count'] + 1)
    features['upper_ratio'] = df['catalog_content'].str.findall(r'[A-Z]').str.len() / (features['text_len'] + 1)
    features['digit_count'] = df['catalog_content'].str.findall(r'\d').str.len()
    features['has_price'] = df['catalog_content'].str.contains(r'\$|price|cost', case=False).astype(int)
    
    for brand in ['sony','samsung','apple','lg','hp','dell','lenovo','nike','adidas']:
        features[f'brand_{brand}'] = df['catalog_content'].str.lower().str.contains(brand).astype(int)
    
    for cat in ['electronic','clothing','book','home','toy','sport','beauty','food']:
        features[f'cat_{cat}'] = df['catalog_content'].str.lower().str.contains(cat).astype(int)
    
    return features

train_advanced = create_features(train_df)
test_advanced = create_features(test_df)
print(f"✅ Advanced features: {train_advanced.shape[1]}\n")

In [ ]:
# Combine features
X_train = pd.concat([train_text_df, train_vision_df, train_advanced], axis=1)
X_test = pd.concat([test_text_df, test_vision_df, test_advanced], axis=1)
y_train = train_df['price'].values

X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print(f"🔥 ACCOUNT 2 Feature Matrix:")
print(f"   Train: {X_train.shape}")
print(f"   Test: {X_test.shape}")
print(f"\n   Text: 2048 (HEAVY - Our advantage!)")
print(f"   Vision: 2048")
print(f"   Advanced: {train_advanced.shape[1]}")
print(f"   TOTAL: {X_train.shape[1]} features\n")

In [ ]:
# Train 3 ML models (45-60 min)
print("🤖 Training 3 ML models with 5-fold CV...")
print("   ⏱️ ETA: 45-60 minutes\n")

def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

models = {
    'xgb': xgb.XGBRegressor(n_estimators=1500, learning_rate=0.03, max_depth=10,
                           subsample=0.8, colsample_bytree=0.8,
                           tree_method='gpu_hist', random_state=42),
    'lgb': lgb.LGBMRegressor(n_estimators=1500, learning_rate=0.03, max_depth=10,
                            subsample=0.8, colsample_bytree=0.8,
                            device='gpu', random_state=42),
    'cat': CatBoostRegressor(iterations=1500, learning_rate=0.03, depth=10,
                            task_type='GPU', verbose=False, random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = {name: np.zeros(len(X_train)) for name in models.keys()}
test_preds = {name: np.zeros(len(X_test)) for name in models.keys()}
cv_scores = {name: [] for name in models.keys()}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"\nFOLD {fold + 1}/5")
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]
    
    for name, model in models.items():
        print(f"  {name}...", end=" ")
        
        if name == 'xgb':
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=100, verbose=False)
        elif name == 'lgb':
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        else:
            model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100, verbose=False)
        
        val_pred = model.predict(X_val)
        oof_preds[name][val_idx] = val_pred
        test_preds[name] += model.predict(X_test) / 5
        
        score = smape(y_val, val_pred)
        cv_scores[name].append(score)
        print(f"{score:.4f}%")
    
    gc.collect()
    torch.cuda.empty_cache()

print("\n✅ Training complete!\n")

In [ ]:
# Create ensembles
simple_avg = sum(test_preds.values()) / len(models)
simple_oof = sum(oof_preds.values()) / len(models)

weights = np.array([1.0 / np.mean(cv_scores[name]) for name in models.keys()])
weights = weights / weights.sum()
weighted_avg = sum(test_preds[name] * w for name, w in zip(models.keys(), weights))
weighted_oof = sum(oof_preds[name] * w for name, w in zip(models.keys(), weights))

sorted_models = sorted(models.keys(), key=lambda x: np.mean(cv_scores[x]))
best_2_avg = sum(test_preds[name] for name in sorted_models[:2]) / 2
best_2_oof = sum(oof_preds[name] for name in sorted_models[:2]) / 2

print("="*60)
print("🏆 ACCOUNT 2 RESULTS (Text-Heavy)")
print("="*60)
print(f"Simple Average:  {smape(y_train, simple_oof):.4f}% CV")
print(f"Weighted Average: {smape(y_train, weighted_oof):.4f}% CV")
print(f"Best 2 Average:  {smape(y_train, best_2_oof):.4f}% CV")

best_score = min(smape(y_train, simple_oof), smape(y_train, weighted_oof), smape(y_train, best_2_oof))
print(f"\n🥇 BEST CV: {best_score:.4f}%")
print(f"📊 vs Phase 5: {57.900 - best_score:.2f}% improvement")
print("="*60 + "\n")

In [ ]:
# Create submissions
print("💾 Creating ACCOUNT 2 submissions...\n")

submissions = {
    'account2_simple_avg': simple_avg,
    'account2_weighted_avg': weighted_avg,
    'account2_best_2_avg': best_2_avg
}

for name, preds in submissions.items():
    df = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': preds})
    df.to_csv(f'{name}.csv', index=False)
    print(f"   ✅ {name}.csv")

print("\n📥 Downloading submissions...\n")
for name in submissions.keys():
    files.download(f'{name}.csv')

print("\n" + "="*60)
print("🎉 ACCOUNT 2 COMPLETE!")
print("="*60)
print("\n📋 Priority: account2_weighted_avg.csv (BEST)")
print("\n🔥 Now run Accounts 1, 3, 4 and blend all results!")